In [1]:
import mediapipe as mp
import cv2
import torch
import torch.nn as nn
import numpy as np
from utils.detection_result_processor import process_result
from utils.image_visualizer import draw_landmarks_on_image

In [2]:
BaseOptions = mp.tasks.BaseOptions
PoseLandmarker = mp.tasks.vision.PoseLandmarker
PoseLandmarkerOptions = mp.tasks.vision.PoseLandmarkerOptions
PoseLandmarkerResult = mp.tasks.vision.PoseLandmarkerResult
VisionRunningMode = mp.tasks.vision.RunningMode
model = "./model/pose_landmarker_lite.task"
options = PoseLandmarkerOptions(base_options=BaseOptions(model_asset_path=model),)



In [4]:
def run_detection():
    frame = 0
    detector = PoseLandmarker.create_from_options(options)
    cap = cv2.VideoCapture(0)
    detection_results = []
    cap.set(cv2.CAP_PROP_FPS,16)
    frame_count = 0
    while cap.isOpened():
            _,frame=  cap.read()
            image = cv2.cvtColor(frame,cv2.COLOR_BGR2RGB)
            image = np.array(image,dtype=np.uint8)
            image = mp.Image(image_format=mp.ImageFormat.SRGB,data=image)
            result = detector.detect(image)
            
            annotated_image = draw_landmarks_on_image(image.numpy_view(),result)
            cv2.imshow("Pose Estimation",cv2.cvtColor(annotated_image,cv2.COLOR_RGB2BGR))
            frame_count +=1
            detection_results.append(process_result(result))
            if frame_count == 16:
                print(detection_results)
                detection_results = []
                
            cv2.waitKey(10)

run_detection()

KeyboardInterrupt: 

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import pandas as pd
from utils.get_root_dir import get_root_dir

class Block(nn.Module):
    def __init__(self,hidden_dim,output_dim):
        self.block = nn.Sequential(
            nn.BatchNorm1d(hidden_dim),
            nn.Linear(hidden_dim,output_dim),
            nn.GELU()
        )
    def forward(self,x):
        return self.block(x)
class ClassificationModel(nn.Module):
    def __init__(self,hidden_dim,num_classes) -> None:
        super().__init__()
        self.in_layers = nn.Sequential(
            nn.Linear(66,hidden_dim),
            nn.GELU(),
            Block(hidden_dim,hidden_dim*2)
        )
        self.bottleneck = nn.Sequential(
            Block(hidden_dim*2,hidden_dim*4),
        )
        self.output_layers = nn.Sequential(
            Block(hidden_dim*4,hidden_dim*2),
            Block(hidden_dim*2,hidden_dim),
            nn.Linear(hidden_dim,num_classes)
        )
        for layer in self.modules():
            if isinstance(layer,nn.Linear):
                nn.init.kaiming_uniform_(layer.weight)
                nn.init.kaiming_uniform_(layer.bias)
    def forward(self,x):
        temp = self.in_layers(x)
        x = self.bottleneck(temp)
        x = self.output_layers[0](x)
        x += temp
        x = self.output_layers[1:](x)
        return x
model = ClassificationModel(300,3)
input_test = torch.rand((1,66))
print(input_test)        

ModuleNotFoundError: No module named 'utils'